In [2]:
print("RAG HR Agent")

RAG HR Agent


In [1]:
import os
from dotenv import load_dotenv

#Langchain libraries for loading
from langchain_community.document_loaders import TextLoader

#Spllitting data
from langchain_text_splitters import RecursiveCharacterTextSplitter

#Embeddings
from langchain_community.embeddings import JinaEmbeddings

#Vector DB
from langchain_community.vectorstores import FAISS

#LLM
from langchain_groq import ChatGroq

#AGENT
from langchain.agents import create_agent

C:\Users\mishr\AppData\Local\Temp\ipykernel_21420\3828315959.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [2]:
load_dotenv()

True

In [2]:
source_data_path = os.path.join("data", "hr_policy.txt")

In [3]:
text_data_loader = TextLoader(source_data_path, encoding="utf-8")

#Langchain process in documents
documents = text_data_loader.load()

print("="*40)
print("Data loaded")
print(documents)


Data loaded
[Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='COMPANY HR POLICY HANDBOOK\nAcme Corp - Employee Handbook (Demo Document)\n\n1. LEAVE POLICY\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\nLeave requests must be submitted through the HR portal at least 5 working days in advance.\nUnused annual leave can be carried forward to the next year, up to a maximum of 5 days.\nSick leave is separate from annual leave, and employees get 10 paid sick days per year.\nA medical certificate is required for sick leave longer than 2 consecutive days.\n\n2. WORK FROM HOME POLICY\nEmployees may work from home up to 2 days per week, subject to manager approval.\nFully remote work arrangements require written approval from the department head.\nEmployees working from home must be reachable during core hours: 10 AM to 4 PM.\n\n3. PROBATION PERIOD\nAll new employees undergo a probation period of 3 months from their date of joining.\nDu

In [8]:
print(documents[0])

page_content='COMPANY HR POLICY HANDBOOK
Acme Corp - Employee Handbook (Demo Document)

1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carried forward to the next year, up to a maximum of 5 days.
Sick leave is separate from annual leave, and employees get 10 paid sick days per year.
A medical certificate is required for sick leave longer than 2 consecutive days.

2. WORK FROM HOME POLICY
Employees may work from home up to 2 days per week, subject to manager approval.
Fully remote work arrangements require written approval from the department head.
Employees working from home must be reachable during core hours: 10 AM to 4 PM.

3. PROBATION PERIOD
All new employees undergo a probation period of 3 months from their date of joining.
During probation, employees are not eligible for paid leave, but may take unpaid leav

In [9]:
total_lengthof_char = len(documents[0].page_content)
print(total_lengthof_char)

2597


In [5]:
#Splitting our data
text_data_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 400,
    chunk_overlap = 50
)

text_chunk = text_data_splitter.split_documents(documents)
print(text_chunk)

[Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='COMPANY HR POLICY HANDBOOK\nAcme Corp - Employee Handbook (Demo Document)'), Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='1. LEAVE POLICY\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\nLeave requests must be submitted through the HR portal at least 5 working days in advance.\nUnused annual leave can be carried forward to the next year, up to a maximum of 5 days.\nSick leave is separate from annual leave, and employees get 10 paid sick days per year.'), Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='A medical certificate is required for sick leave longer than 2 consecutive days.'), Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='2. WORK FROM HOME POLICY\nEmployees may work from home up to 2 days per week, subject to manager approval.\nFully remote work arrangements require written approval from the department head.\nE

In [11]:
print(text_chunk[8].page_content)

7. HOLIDAYS
The company observes 12 public holidays every year, as per the official holiday calendar
published by HR at the start of each year.
Employees working on a public holiday are eligible for compensatory leave.


In [6]:
embeddings_model = JinaEmbeddings(model_name="jina-embeddings-v2-base-en")

vector_store = FAISS.from_documents(text_chunk, embeddings_model)

print("Chunks stored", vector_store.index.ntotal)



Chunks stored 10


In [7]:
test_query = "How many sick leaves employees get"

top_matches = vector_store.similarity_search(test_query, k=2 )

for i, match in enumerate(top_matches, start=1):
    print(f"-----Match{i}-----")
    print(match.page_content)

-----Match1-----
1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carried forward to the next year, up to a maximum of 5 days.
Sick leave is separate from annual leave, and employees get 10 paid sick days per year.
-----Match2-----
7. HOLIDAYS
The company observes 12 public holidays every year, as per the official holiday calendar
published by HR at the start of each year.
Employees working on a public holiday are eligible for compensatory leave.


In [8]:
llm = ChatGroq(
    model = "openai/gpt-oss-120b",
    temperature=0
)

print(llm.model_name)

openai/gpt-oss-120b


In [9]:
test_response = llm.invoke("Can you tell me top 3 best LLMs as of today?")

In [11]:
print(test_response.content)

Sure! As of August 2026, the three most widely‑regarded “best” large language models (LLMs) in terms of overall performance, versatility, and accessibility are:

| Rank | Model (Provider) | Key Strengths | Typical Use‑Cases |
|------|------------------|----------------|-------------------|
| **1** | **Gemini 2 Ultra** (Google DeepMind) | • 1.1 trillion parameters (mix of dense and sparsely‑gated experts)  <br>• State‑of‑the‑art on most benchmark suites (MMLU, BIG‑Bench, HELM)  <br>• Strong multimodal reasoning (text + image + video)  <br>• Built‑in safety and factuality layers that reduce hallucinations by ~45 % vs. previous generation | • Enterprise AI assistants, RAG‑powered search, code generation, research‑grade reasoning, multimodal content creation |
| **2** | **LLaMA 3‑Chat** (Meta AI) | • 2 trillion parameters (dense) with a highly optimized transformer kernel  <br>• Open‑source licensing (research‑friendly, commercial‑permissive)  <br>• Excellent zero‑shot and few‑shot perform

CREATING THE TOOL

In [13]:
#Creating the retriever from the vector_store variable which contains the FAISS vector DB
retriever = vector_store.as_retriever(search_keargs = {"k" : 3})


#Creating a tool (function)
def search_hr_policy(question : str) -> str:
    """
     Search the HR policy document for information about leave, work from home,
    probation, notice period, reimbursement, code of conduct, holidays, or exit process.

    """
    matching_chunks = retriever.invoke(question)

    return "\n\n".join(chunk.page_content for chunk in matching_chunks)

CREATING AGENT

In [14]:
hr_assistant = create_agent(
    llm,
    tools = [search_hr_policy],
    system_prompt="""
    
    You are a friendly HR assistant working for Acme Crop. 
    Always use the search_hr_policy tool to look up 
    facts before answering. 
    If the answer isn't in the search results, say you don't know 
    instead of guessing.

"""
)

In [19]:
def ask_hr_assistant(question: str) -> str:
    print("="*80)
    print("QUESTION:", question)
    print("="*80)

    response = hr_assistant.invoke(
        {
            "messages": [
                    {   
                        "role": "user", 
                        "content": question
                    }
            ]
        }
    )
    answer = response["messages"][-1].content

    print("ANSWER:", answer)
    print("="*80)
    print()



In [20]:
ask_hr_assistant("Who are you?")

QUESTION: Who are you?
ANSWER: I’m your friendly HR assistant here at Acme Crop. I’m happy to help you with any questions you have about our HR policies, benefits, procedures, or anything else related to working at Acme Crop. Just let me know what you need!



In [21]:
ask_hr_assistant("Tell me how to apply for leave")

QUESTION: Tell me how to apply for leave
ANSWER: Sure! Here’s how you can apply for leave at Acme Crop:

1. **Check Your Leave Balance**  
   - Full‑time employees have **20 days of paid annual leave** per calendar year (plus 10 paid sick days). Unused annual leave can be carried forward up to **5 days**.

2. **Plan Ahead**  
   - Submit your request **at least 5 working days before the start date** of the leave (unless it’s an emergency or sick leave).

3. **Use the HR Portal**  
   - Log in to the **HR portal** (the same system you use for timesheets and other HR requests).  
   - Navigate to the **“Leave Request”** section.

4. **Enter the Details**  
   - **Leave type** (e.g., Annual Leave, Sick Leave).  
   - **Start and end dates** (the system will automatically calculate the number of working days).  
   - **Reason** (optional for annual leave, required for sick leave).  
   - Attach any supporting documents if needed (e.g., a medical certificate for sick leave).

5. **Submit fo